In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [5]:
DATA_PATH = r"D:\Telco_Customer_Churn\data\processed_data\telco_cleaned.csv"
FIGURES_DIR = r"D:\Telco_Customer_Churn\reports\figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(3)

Dataset loaded: 7043 rows x 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


| Category        | Features                                                                                                                                | Business Meaning                                                                                      |
| --------------- | --------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- |
| **Demographic** | Gender, SeniorCitizen, Partner, Dependents                                                                                              | Describes the customer's personal and household characteristics.                                      |
| **Account**     | Tenure, Contract, PaperlessBilling, PaymentMethod                                                                                       | Describes the customer's relationship with the company, billing preferences, and contract commitment. |
| **Services**    | PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies | Describes the telecom products and value-added services subscribed by the customer.                   |
| **Financial**   | MonthlyCharges, TotalCharges                                                                                                            | Represents the customer's spending and revenue contribution to the company.                           |
| **Target**      | Churn                                                                                                                                   | Indicates whether the customer left the company.                                                      |


# Feature Engineering

## Objective

The objective of feature engineering is to transform raw customer information into more meaningful business features that can improve machine learning model performance.

Rather than relying only on the original dataset, we create additional features that better represent customer behavior, engagement, commitment, and value.

Each engineered feature is created based on business logic and will later be evaluated for its contribution to churn prediction.

In [9]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

## Feature 1: TotalServicesSubscribed

In [19]:
# Create TotalServicesSubscribed feature
service_columns = [
    'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies'
]

df['TotalServicesSubscribed'] = 0

In [20]:
# Phone Service
df['TotalServicesSubscribed'] += (df['PhoneService'] == 'Yes').astype(int)

# Mulitple lines
df['TotalServicesSubscribed'] += (df['MultipleLines'] == 'Yes').astype(int)

# Internet services
df['TotalServicesSubscribed'] += (df['InternetService'].isin(['DSL', 'Fiber optic'])).astype(int)

#Internet-based services
internet_services = [
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

for col in internet_services:
    df['TotalServicesSubscribed'] += (df[col] == 'Yes').astype(int)

In [21]:
df['TotalServicesSubscribed'].describe()

count    7043.000000
mean        4.146244
std         2.312720
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max         9.000000
Name: TotalServicesSubscribed, dtype: float64

In [22]:
df['TotalServicesSubscribed'].value_counts().sort_index()

TotalServicesSubscribed
1    1264
2     859
3     846
4     965
5     922
6     908
7     676
8     395
9     208
Name: count, dtype: int64

In [23]:
df['TotalServicesSubscribed']

0       2
1       4
2       4
3       4
4       2
       ..
7038    8
7039    7
7040    2
7041    3
7042    7
Name: TotalServicesSubscribed, Length: 7043, dtype: int64

## Feature 2: CustomerTenureGroup

In [25]:
def tenure_group(tenure):
    if tenure <= 12:
        return "New Customer"
    elif tenure <= 48:
        return "Regular Customer"
    else:
        return "Loyal Customer"

df['CustomerTenureGroup'] =  df['tenure'].apply(tenure_group)

In [26]:
df['CustomerTenureGroup'].value_counts()

CustomerTenureGroup
Regular Customer    2618
Loyal Customer      2239
New Customer        2186
Name: count, dtype: int64

## Feature 3: CustomerValueTier

In [27]:
q1 = df['TotalCharges'].quantile(0.33)
q2 = df['TotalCharges'].quantile(0.66)

In [28]:
def customer_value(TotalCharges):
    if TotalCharges <= q1:
        return "Low Value"
    elif TotalCharges <= q2:
        return "Medium Value"
    else:
        return "High Value"

df['CustomerValueTier'] = df['TotalCharges'].apply(customer_value)

In [29]:
df['CustomerValueTier'].value_counts()

CustomerValueTier
High Value      2395
Low Value       2324
Medium Value    2324
Name: count, dtype: int64

## Feature 4: DigitalAdoptionScore

In [30]:
digital_columns = [
    'PaperlessBilling',
    'OnlineBackup',
    'OnlineSecurity',
    'StreamingTV',
    'StreamingMovies'
]

df['DigitalAdoptionScore'] = 0

In [31]:
df['DigitalAdoptionScore'] +=(df['PaperlessBilling'] == 'Yes').astype(int)

for col in digital_columns[1:]:
    df['DigitalAdoptionScore'] +=(df[col] == 'Yes').astype(int)

In [32]:
df['DigitalAdoptionScore'].describe()

count    7043.000000
mean        1.996024
std         1.484842
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         5.000000
Name: DigitalAdoptionScore, dtype: float64

In [33]:
df['DigitalAdoptionScore'].value_counts().sort_index()

DigitalAdoptionScore
0    1417
1    1557
2    1359
3    1373
4    1021
5     316
Name: count, dtype: int64

## Feature Engineering Summary

In [34]:
new_features = [
    'TotalServicesSubscribed',
    'CustomerTenureGroup',
    'CustomerValueTier',
    'DigitalAdoptionScore'
]

df[new_features].head()

,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,2,New Customer,Low Value,2
1,4,Regular Customer,Medium Value,1
2,4,New Customer,Low Value,3
3,4,Regular Customer,Medium Value,1
4,2,New Customer,Low Value,1


In [35]:
df[new_features].isnull().sum()

TotalServicesSubscribed    0
CustomerTenureGroup        0
CustomerValueTier          0
DigitalAdoptionScore       0
dtype: int64

### Save the Feature-Engineered Dataset

In [38]:
PROCESSED_DIR  = r"D:\Telco_Customer_Churn\data\processed_data"
OUTPUT_FILE    = os.path.join(PROCESSED_DIR, "churn_feature_engineered.csv")

# Create directory if it doesn't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Save
df.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset saved to:")
print(f"  {OUTPUT_FILE}")
print(f"  Shape : {df.shape}")
print(f"  Size  : {os.path.getsize(OUTPUT_FILE) / 1024:.1f} KB")

Cleaned dataset saved to:
  D:\Telco_Customer_Churn\data\processed_data\churn_feature_engineered.csv
  Shape : (7043, 25)
  Size  : 1165.5 KB
